Cell 1   Imports
Cell 2   Load processed Parquet
Cell 3   Create V10 features
Cell 4   Create train/test split
Cell 5   Train Stage 1 classifier
Cell 6   Train Stage 2 log1p regressor
Cell 7   Generate V12 predictions
Cell 8   March evaluation
Cell 9   April evaluation
Cell 10  Compare results
Cell 11  Save models
Cell 12  Inventory optimization

In [1]:
import pandas as pd
import numpy as np
import duckdb

from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

import joblib
from pathlib import Path

In [2]:
DATA_PATH = "../data/processed/sales_processed.parquet"
MODEL_DIR = Path("../models")

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Data path:", DATA_PATH)
print("Model directory:", MODEL_DIR)

Data path: ../data/processed/sales_processed.parquet
Model directory: ..\models


In [3]:
con = duckdb.connect()

query = """
SELECT
    item_id,
    dept_id,
    cat_id,
    store_id,
    state_id,
    date,
    sales,
    dayofweek,
    month,
    week,
    event_name_1,
    event_type_1,
    event_name_2,
    event_type_2,
    snap_CA,
    snap_TX,
    snap_WI
FROM read_parquet(?)
WHERE item_id IN (
    'HOBBIES_1_001',
    'HOBBIES_1_002',
    'HOBBIES_1_003',
    'HOBBIES_1_004',
    'HOBBIES_1_005'
)
AND state_id IN ('CA', 'TX', 'WI')
"""

result_base = con.execute(
    query,
    [DATA_PATH]
).df()

print("Shape:", result_base.shape)
print(result_base.head())

Shape: (95650, 17)
         item_id    dept_id   cat_id store_id state_id       date  sales  \
0  HOBBIES_1_001  HOBBIES_1  HOBBIES     WI_2       WI 2011-09-26      0   
1  HOBBIES_1_002  HOBBIES_1  HOBBIES     WI_2       WI 2011-09-26      0   
2  HOBBIES_1_003  HOBBIES_1  HOBBIES     WI_2       WI 2011-09-26      0   
3  HOBBIES_1_004  HOBBIES_1  HOBBIES     WI_2       WI 2011-09-26      1   
4  HOBBIES_1_005  HOBBIES_1  HOBBIES     WI_2       WI 2011-09-26      1   

   dayofweek  month  week event_name_1 event_type_1 event_name_2 event_type_2  \
0          0      9    39          NaN          NaN          NaN          NaN   
1          0      9    39          NaN          NaN          NaN          NaN   
2          0      9    39          NaN          NaN          NaN          NaN   
3          0      9    39          NaN          NaN          NaN          NaN   
4          0      9    39          NaN          NaN          NaN          NaN   

   snap_CA  snap_TX  snap_WI  
0     

In [4]:
result_base['date']=pd.to_datetime(
    result_base['date']
)
result_base["weekofyear"]=(
    result_base["date"].dt.isocalendar().week.astype(int)
)
result_base["is_weekend"]=(
    result_base["weekofyear"]>=5
).astype(int)
result_base["has_event"]=(
    result_base["event_name_1"].notna()
    |
    result_base["event_name_2"].notna()
).astype(int)


In [5]:
result_base=(
    result_base.sort_values(["item_id","store_id","date"]).copy()
)
print(
    result_base[
        ["item_id", "store_id", "date", "sales"]
    ].head(10)
)

            item_id store_id       date  sales
6135  HOBBIES_1_001     CA_1 2011-01-29      0
6185  HOBBIES_1_001     CA_1 2011-01-30      0
6235  HOBBIES_1_001     CA_1 2011-01-31      0
6285  HOBBIES_1_001     CA_1 2011-02-01      0
6335  HOBBIES_1_001     CA_1 2011-02-02      0
6385  HOBBIES_1_001     CA_1 2011-02-03      0
6435  HOBBIES_1_001     CA_1 2011-02-04      0
6485  HOBBIES_1_001     CA_1 2011-02-05      0
6535  HOBBIES_1_001     CA_1 2011-02-06      0
6585  HOBBIES_1_001     CA_1 2011-02-07      0


In [6]:
GROUP_COLS = [
    "item_id",
    "store_id"
]

for lag in [1, 7, 14, 28]:
    result_base[f"lag_{lag}"] = (
        result_base
        .groupby(GROUP_COLS)["sales"]
        .shift(lag)
    )

In [7]:
result_base["rolling_mean_7"] = (
    result_base
    .groupby(GROUP_COLS)["sales"]
    .transform(
        lambda x:
        x.shift(1).rolling(7).mean()
    )
)

result_base["rolling_mean_28"] = (
    result_base
    .groupby(GROUP_COLS)["sales"]
    .transform(
        lambda x:
        x.shift(1).rolling(28).mean()
    )
)

result_base["rolling_max_7"] = (
    result_base
    .groupby(GROUP_COLS)["sales"]
    .transform(
        lambda x:
        x.shift(1).rolling(7).max()
    )
)

result_base["rolling_max_28"] = (
    result_base
    .groupby(GROUP_COLS)["sales"]
    .transform(
        lambda x:
        x.shift(1).rolling(28).max()
    )
)

result_base["rolling_std_7"] = (
    result_base
    .groupby(GROUP_COLS)["sales"]
    .transform(
        lambda x:
        x.shift(1).rolling(7).std()
    )
)

In [8]:
result_base["trend"] = (
    result_base["rolling_mean_7"]
    -
    result_base["rolling_mean_28"]
)

In [9]:
result_base["state_encoded"] = (
    result_base["state_id"]
    .astype("category")
    .cat.codes
)

result_base["store_encoded"] = (
    result_base["store_id"]
    .astype("category")
    .cat.codes
)

result_base["item_encoded"] = (
    result_base["item_id"]
    .astype("category")
    .cat.codes
)

In [10]:
FEATURES = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "rolling_max_7",
    "rolling_max_28",
    "rolling_std_7",
    "trend",
    "dayofweek",
    "month",
    "weekofyear",
    "is_weekend",
    "has_event",
    "state_encoded",
    "store_encoded",
    "item_encoded"
]

print("Number of features:", len(FEATURES))

Number of features: 18


In [11]:
result = result_base.dropna(
    subset=FEATURES
).copy()

print("Final dataset shape:", result.shape)

missing_features = [
    col for col in FEATURES
    if col not in result.columns
]

print("Missing features:", missing_features)

Final dataset shape: (94250, 33)
Missing features: []


In [12]:
train_april = result[
    result["date"] < "2016-04-01"
].copy()

test_april = result[
    (result["date"] >= "2016-04-01")
    &
    (result["date"] <= "2016-04-24")
].copy()

print("April train:", train_april.shape)
print("April test:", test_april.shape)

print(
    "April test:",
    test_april["date"].min(),
    "to",
    test_april["date"].max()
)

April train: (93050, 33)
April test: (1200, 33)
April test: 2016-04-01 00:00:00 to 2016-04-24 00:00:00


In [13]:
X_train_v10 = train_april[FEATURES]
y_train_v10 = train_april["sales"]

X_test_v10 = test_april[FEATURES]
y_test_v10 = test_april["sales"]

model_v10 = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model_v10.fit(
    X_train_v10,
    y_train_v10
)

pred_v10 = model_v10.predict(
    X_test_v10
)

pred_v10 = np.clip(
    pred_v10,
    0,
    None
)

In [14]:
v10_april_mae = mean_absolute_error(
    y_test_v10,
    pred_v10
)

v10_april_rmse = mean_squared_error(
    y_test_v10,
    pred_v10
) ** 0.5

print("V10 April MAE:", v10_april_mae)
print("V10 April RMSE:", v10_april_rmse)

V10 April MAE: 0.6839123964309692
V10 April RMSE: 1.0356415960332344


In [15]:
train_v12 = train_april.copy()
test_v12 = test_april.copy()

train_v12["has_demand_target"] = (
    train_v12["sales"] > 0
).astype(int)

test_v12["has_demand_target"] = (
    test_v12["sales"] > 0
).astype(int)

In [16]:
X_train_cls = train_v12[FEATURES]
y_train_cls = train_v12["has_demand_target"]

X_test_cls = test_v12[FEATURES]
y_test_cls = test_v12["has_demand_target"]

demand_classifier = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss"
)

demand_classifier.fit(
    X_train_cls,
    y_train_cls
)

demand_prediction = demand_classifier.predict(
    X_test_cls
)

In [17]:
classifier_accuracy = accuracy_score(
    y_test_cls,
    demand_prediction
)

classifier_precision = precision_score(
    y_test_cls,
    demand_prediction,
    zero_division=0
)

classifier_recall = recall_score(
    y_test_cls,
    demand_prediction,
    zero_division=0
)

classifier_f1 = f1_score(
    y_test_cls,
    demand_prediction,
    zero_division=0
)

print("Accuracy :", classifier_accuracy)
print("Precision:", classifier_precision)
print("Recall   :", classifier_recall)
print("F1       :", classifier_f1)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_cls,
        demand_prediction
    )
)

Accuracy : 0.7191666666666666
Precision: 0.725925925925926
Recall   : 0.42701525054466233
F1       : 0.5377229080932785

Confusion Matrix:
[[667  74]
 [263 196]]


In [18]:
train_quantity = train_v12[
    train_v12["sales"] > 0
].copy()

X_train_quantity = train_quantity[
    FEATURES
]

y_train_quantity = np.log1p(
    train_quantity["sales"]
)

print(
    "Positive-demand rows:",
    len(train_quantity)
)

Positive-demand rows: 27088


In [19]:
quantity_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

quantity_model.fit(
    X_train_quantity,
    y_train_quantity
)

,"objective objective: str | xgboost.objective.Objective | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGB

In [20]:
quantity_prediction_log = (
    quantity_model.predict(
        test_v12[FEATURES]
    )
)

quantity_prediction = np.expm1(
    quantity_prediction_log
)

quantity_prediction = np.clip(
    quantity_prediction,
    0,
    None
)

In [21]:
final_prediction = (
    quantity_prediction
    *
    demand_prediction
)

test_v12["prediction"] = final_prediction

In [22]:
v12_april_mae = mean_absolute_error(
    test_v12["sales"],
    test_v12["prediction"]
)

v12_april_rmse = mean_squared_error(
    test_v12["sales"],
    test_v12["prediction"]
) ** 0.5

print("V12 April MAE :", v12_april_mae)
print("V12 April RMSE:", v12_april_rmse)

V12 April MAE : 0.6210867561896642
V12 April RMSE: 1.1444392213109273


In [23]:
train_march = result[
    result["date"] < "2016-03-04"
].copy()

test_march = result[
    (result["date"] >= "2016-03-04")
    &
    (result["date"] <= "2016-03-31")
].copy()

print("March train:", train_march.shape)
print("March test:", test_march.shape)

print(
    "March test:",
    test_march["date"].min(),
    "to",
    test_march["date"].max()
)

March train: (91650, 33)
March test: (1400, 33)
March test: 2016-03-04 00:00:00 to 2016-03-31 00:00:00


In [24]:
train_march = result[
    result["date"] < "2016-03-04"
].copy()

test_march = result[
    (result["date"] >= "2016-03-04")
    &
    (result["date"] <= "2016-03-31")
].copy()

print("March train:", train_march.shape)
print("March test:", test_march.shape)

print(
    "March test:",
    test_march["date"].min(),
    "to",
    test_march["date"].max()
)

March train: (91650, 33)
March test: (1400, 33)
March test: 2016-03-04 00:00:00 to 2016-03-31 00:00:00


In [25]:
train_march["has_demand_target"] = (
    train_march["sales"] > 0
).astype(int)

test_march["has_demand_target"] = (
    test_march["sales"] > 0
).astype(int)

X_train_cls_march = train_march[
    FEATURES
]

y_train_cls_march = train_march[
    "has_demand_target"
]

X_test_cls_march = test_march[
    FEATURES
]

y_test_cls_march = test_march[
    "has_demand_target"
]

demand_classifier_march = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss"
)

demand_classifier_march.fit(
    X_train_cls_march,
    y_train_cls_march
)

demand_prediction_march = (
    demand_classifier_march.predict(
        X_test_cls_march
    )
)

In [26]:
train_quantity_march = train_march[
    train_march["sales"] > 0
].copy()

X_train_quantity_march = (
    train_quantity_march[FEATURES]
)

y_train_quantity_march = np.log1p(
    train_quantity_march["sales"]
)

quantity_model_march = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

quantity_model_march.fit(
    X_train_quantity_march,
    y_train_quantity_march
)

quantity_prediction_log_march = (
    quantity_model_march.predict(
        test_march[FEATURES]
    )
)

quantity_prediction_march = np.expm1(
    quantity_prediction_log_march
)

quantity_prediction_march = np.clip(
    quantity_prediction_march,
    0,
    None
)

In [27]:
final_prediction_march = (
    quantity_prediction_march
    *
    demand_prediction_march
)

test_march["prediction"] = (
    final_prediction_march
)

In [28]:
v12_march_mae = mean_absolute_error(
    test_march["sales"],
    test_march["prediction"]
)

v12_march_rmse = mean_squared_error(
    test_march["sales"],
    test_march["prediction"]
) ** 0.5

print("V12 March MAE :", v12_march_mae)
print("V12 March RMSE:", v12_march_rmse)

V12 March MAE : 0.6658740660122463
V12 March RMSE: 1.209509018811185


In [29]:
test_april["naive_prediction"] = (
    test_april["lag_1"]
)

test_march["naive_prediction"] = (
    test_march["lag_1"]
)

In [30]:
naive_april_mae = mean_absolute_error(
    test_april["sales"],
    test_april["naive_prediction"]
)

naive_april_rmse = mean_squared_error(
    test_april["sales"],
    test_april["naive_prediction"]
) ** 0.5

naive_march_mae = mean_absolute_error(
    test_march["sales"],
    test_march["naive_prediction"]
)

naive_march_rmse = mean_squared_error(
    test_march["sales"],
    test_march["naive_prediction"]
) ** 0.5

print("April naive MAE :", naive_april_mae)
print("April naive RMSE:", naive_april_rmse)

print()

print("March naive MAE :", naive_march_mae)
print("March naive RMSE:", naive_march_rmse)

April naive MAE : 0.8125
April naive RMSE: 1.4453949863849214

March naive MAE : 0.8442857142857143
March naive RMSE: 1.5142183273415835


In [31]:
evaluation = pd.DataFrame({
    "Period": [
        "March",
        "April"
    ],
    "Naive MAE": [
        naive_march_mae,
        naive_april_mae
    ],
    "V12 MAE": [
        v12_march_mae,
        v12_april_mae
    ],
    "Naive RMSE": [
        naive_march_rmse,
        naive_april_rmse
    ],
    "V12 RMSE": [
        v12_march_rmse,
        v12_april_rmse
    ]
})

evaluation

,Period,Naive MAE,V12 MAE,Naive RMSE,V12 RMSE
0,March,0.844286,0.665874,1.514218,1.209509
1,April,0.812500,0.621087,1.445395,1.144439


In [32]:
evaluation["MAE Improvement %"] = (
    (
        evaluation["Naive MAE"]
        -
        evaluation["V12 MAE"]
    )
    /
    evaluation["Naive MAE"]
    *
    100
)

evaluation["RMSE Improvement %"] = (
    (
        evaluation["Naive RMSE"]
        -
        evaluation["V12 RMSE"]
    )
    /
    evaluation["Naive RMSE"]
    *
    100
)

evaluation

,Period,Naive MAE,V12 MAE,Naive RMSE,V12 RMSE,MAE Improvement %,RMSE Improvement %
0,March,0.844286,0.665874,1.514218,1.209509,21.131667,20.123208
1,April,0.812500,0.621087,1.445395,1.144439,23.558553,20.821697


In [33]:
joblib.dump(
    demand_classifier,
    MODEL_DIR / "v12_demand_classifier.pkl"
)

joblib.dump(
    quantity_model,
    MODEL_DIR / "v12_quantity_model.pkl"
)

joblib.dump(
    FEATURES,
    MODEL_DIR / "v12_features.pkl"
)

print("V12 models and feature list saved.")

V12 models and feature list saved.


In [34]:
evaluation.to_csv(
    MODEL_DIR / "v12_evaluation.csv",
    index=False
)

print("Evaluation saved.")

Evaluation saved.


In [35]:
inventory_stats = (
    train_march
    .groupby(
        ["item_id", "store_id"]
    )["sales"]
    .agg(
        avg_daily_demand="mean",
        demand_std="std"
    )
    .reset_index()
)

lead_time_days = 7

z_score = 1.645

inventory_stats["lead_time_demand"] = (
    inventory_stats["avg_daily_demand"]
    *
    lead_time_days
)

inventory_stats["safety_stock"] = (
    z_score
    *
    inventory_stats["demand_std"]
    *
    np.sqrt(lead_time_days)
)

inventory_stats["reorder_point"] = (
    inventory_stats["lead_time_demand"]
    +
    inventory_stats["safety_stock"]
)

inventory_stats.head()

,item_id,store_id,avg_daily_demand,demand_std,lead_time_demand,safety_stock,reorder_point
0,HOBBIES_1_001,CA_1,0.298963,0.666644,2.092744,2.901411,4.994155
1,HOBBIES_1_001,CA_2,0.273322,0.667421,1.913257,2.904790,4.818047
2,HOBBIES_1_001,CA_3,0.380797,0.804668,2.665576,3.502127,6.167703
3,HOBBIES_1_001,CA_4,0.322968,0.700438,2.260775,3.048488,5.309262
4,HOBBIES_1_001,TX_1,0.132024,0.423163,0.924168,1.841715,2.765883


In [36]:
print(test_april.columns.tolist())

['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'date', 'sales', 'dayofweek', 'month', 'week', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'weekofyear', 'is_weekend', 'has_event', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_28', 'rolling_max_7', 'rolling_max_28', 'rolling_std_7', 'trend', 'state_encoded', 'store_encoded', 'item_encoded', 'naive_prediction']


In [39]:
forecast_summary = (
    test_april
    .groupby(
        ["item_id", "store_id"]
    )
    .agg(
        forecast_demand=(
            "naive_prediction",
            "sum"
        ),
        actual_demand=(
            "sales",
            "sum"
        )
    )
    .reset_index()
)

forecast_summary.head()

,item_id,store_id,forecast_demand,actual_demand
0,HOBBIES_1_001,CA_1,25.0,26
1,HOBBIES_1_001,CA_2,15.0,17
2,HOBBIES_1_001,CA_3,24.0,27
3,HOBBIES_1_001,CA_4,13.0,13
4,HOBBIES_1_001,TX_1,6.0,8


In [40]:
forecast_summary = forecast_summary.merge(
    inventory_stats,
    on=["item_id", "store_id"],
    how="left"
)

forecast_summary.head()

,item_id,store_id,forecast_demand,actual_demand,avg_daily_demand,demand_std,lead_time_demand,safety_stock,reorder_point
0,HOBBIES_1_001,CA_1,25.0,26,0.298963,0.666644,2.092744,2.901411,4.994155
1,HOBBIES_1_001,CA_2,15.0,17,0.273322,0.667421,1.913257,2.904790,4.818047
2,HOBBIES_1_001,CA_3,24.0,27,0.380797,0.804668,2.665576,3.502127,6.167703
3,HOBBIES_1_001,CA_4,13.0,13,0.322968,0.700438,2.260775,3.048488,5.309262
4,HOBBIES_1_001,TX_1,6.0,8,0.132024,0.423163,0.924168,1.841715,2.765883


In [41]:
forecast_summary["recommended_stock"] = (
    forecast_summary["forecast_demand"]
    +
    forecast_summary["safety_stock"]
)

forecast_summary["recommended_stock"] = (
    np.ceil(
        forecast_summary["recommended_stock"]
    )
)

forecast_summary[
    [
        "item_id",
        "store_id",
        "forecast_demand",
        "safety_stock",
        "reorder_point",
        "recommended_stock"
    ]
].head(20)

,item_id,store_id,forecast_demand,safety_stock,reorder_point,recommended_stock
0,HOBBIES_1_001,CA_1,25.0,2.901411,4.994155,28.0
1,HOBBIES_1_001,CA_2,15.0,2.904790,4.818047,18.0
2,HOBBIES_1_001,CA_3,24.0,3.502127,6.167703,28.0
3,HOBBIES_1_001,CA_4,13.0,3.048488,5.309262,17.0
4,HOBBIES_1_001,TX_1,6.0,1.841715,2.765883,8.0
5,HOBBIES_1_001,TX_2,8.0,2.338099,3.812185,11.0
6,HOBBIES_1_001,TX_3,8.0,2.181632,3.575522,11.0
7,HOBBIES_1_001,WI_1,7.0,1.481340,2.065628,9.0
8,HOBBIES_1_001,WI_2,5.0,1.297233,1.705853,7.0
9,HOBBIES_1_001,WI_3,2.0,1.689119,2.491083,4.0


In [42]:
forecast_summary.to_csv(
    MODEL_DIR / "inventory_recommendations.csv",
    index=False
)

print(
    "Inventory recommendations saved."
)

Inventory recommendations saved.


In [43]:
print("========== PROJECT RESULTS ==========")

print(
    f"March V12 MAE : {v12_march_mae:.4f}"
)

print(
    f"March V12 RMSE: {v12_march_rmse:.4f}"
)

print(
    f"April V12 MAE : {v12_april_mae:.4f}"
)

print(
    f"April V12 RMSE: {v12_april_rmse:.4f}"
)

print("\nSaved models:")

print(
    MODEL_DIR / "v12_demand_classifier.pkl"
)

print(
    MODEL_DIR / "v12_quantity_model.pkl"
)

print(
    MODEL_DIR / "v12_features.pkl"
)

print("\nDone.")

========== PROJECT RESULTS ==========
March V12 MAE : 0.6659
March V12 RMSE: 1.2095
April V12 MAE : 0.6211
April V12 RMSE: 1.1444

Saved models:
..\models\v12_demand_classifier.pkl
..\models\v12_quantity_model.pkl
..\models\v12_features.pkl

Done.


In [44]:
forecast_summary["stockout"] = (
    forecast_summary["actual_demand"]
    > forecast_summary["recommended_stock"]
).astype(int)

forecast_summary["excess_inventory"] = (
    forecast_summary["recommended_stock"]
    - forecast_summary["actual_demand"]
).clip(lower=0)

forecast_summary["shortage"] = (
    forecast_summary["actual_demand"]
    - forecast_summary["recommended_stock"]
).clip(lower=0)

In [45]:
service_level = 1 - forecast_summary["stockout"].mean()

average_excess = forecast_summary["excess_inventory"].mean()

average_shortage = forecast_summary["shortage"].mean()

print("Service Level:", service_level)
print("Average Excess Inventory:", average_excess)
print("Average Shortage:", average_shortage)

Service Level: 1.0
Average Excess Inventory: 3.88
Average Shortage: 0.0


NOW WE WILL TRY TO OPTIMIZE OUT INVENTRY


# Our supplier lead time is assumed to be 7 days.
# This means a new order placed today is expected to arrive
# after 7 days.
lead_time_days = 7

# We use a z-score of 1.645, corresponding to approximately
# a 95% one-sided service-level target under the normal-demand
# assumption.
z_score = 1.645


# ------------------------------------------------------------
# 1. Calculate historical demand statistics
# ------------------------------------------------------------

# Calculate the average daily demand and demand variability
# for each product-store combination using training data only.
inventory_stats = (
    train_april
    .groupby(["item_id", "store_id"])["sales"]
    .agg(
        avg_daily_demand="mean",
        demand_std="std"
    )
    .reset_index()
)


# ------------------------------------------------------------
# 2. Calculate expected demand during lead time
# ------------------------------------------------------------

# Estimate how many units are expected to be sold while
# waiting for a new order to arrive.
inventory_stats["lead_time_demand"] = (
    inventory_stats["avg_daily_demand"] * lead_time_days
)


# ------------------------------------------------------------
# 3. Calculate safety stock
# ------------------------------------------------------------

# Safety stock provides a buffer against uncertainty in demand.
# Higher demand variability results in higher safety stock.
inventory_stats["safety_stock"] = (
    z_score
    * inventory_stats["demand_std"]
    * np.sqrt(lead_time_days)
)


# ------------------------------------------------------------
# 4. Calculate reorder point
# ------------------------------------------------------------

# Reorder point represents the inventory level at which
# a new order should be placed.
#
# If current inventory falls below this level, the store
# should reorder because the remaining stock is expected
# to cover approximately the lead-time demand plus a
# safety buffer.
inventory_stats["reorder_point"] = (
    inventory_stats["lead_time_demand"]
    + inventory_stats["safety_stock"]
)

inventory_stats.head()

In [47]:
test_april = test_april.sort_values(
    ["item_id", "store_id", "date"]
).copy()

test_april["actual_next_7_days"] = (
    test_april
    .groupby(["item_id", "store_id"])["sales"]
    .transform(
        lambda x: x.shift(-1)
                  .rolling(7, min_periods=7)
                  .sum()
                  .shift(-6)
    )
)

print(
    test_april[
        [
            "item_id",
            "store_id",
            "date",
            "sales",
            "actual_next_7_days"
        ]
    ].head(15)
)

             item_id store_id       date  sales  actual_next_7_days
95525  HOBBIES_1_001     CA_1 2016-04-01      0                10.0
95575  HOBBIES_1_001     CA_1 2016-04-02      0                11.0
95625  HOBBIES_1_001     CA_1 2016-04-03      1                12.0
90880  HOBBIES_1_001     CA_1 2016-04-04      0                12.0
90930  HOBBIES_1_001     CA_1 2016-04-05      4                 8.0
90980  HOBBIES_1_001     CA_1 2016-04-06      2                 6.0
91030  HOBBIES_1_001     CA_1 2016-04-07      3                 4.0
91080  HOBBIES_1_001     CA_1 2016-04-08      0                 5.0
91130  HOBBIES_1_001     CA_1 2016-04-09      1                 7.0
91180  HOBBIES_1_001     CA_1 2016-04-10      2                 5.0
91230  HOBBIES_1_001     CA_1 2016-04-11      0                 6.0
91280  HOBBIES_1_001     CA_1 2016-04-12      0                 7.0
91330  HOBBIES_1_001     CA_1 2016-04-13      0                 8.0
91380  HOBBIES_1_001     CA_1 2016-04-14      1 

In [51]:
print("V12 predictions:", len(final_prediction))
print("test_april:", len(test_april))

V12 predictions: 1200
test_april: 1200


In [54]:
test_april["prediction"] = final_prediction

print(
    test_april[
        [
            "item_id",
            "store_id",
            "date",
            "sales",
            "prediction",
            "actual_next_7_days"
        ]
    ].head(15)
)

             item_id store_id       date  sales  prediction  \
95525  HOBBIES_1_001     CA_1 2016-04-01      0    1.385654   
95575  HOBBIES_1_001     CA_1 2016-04-02      0    1.385929   
95625  HOBBIES_1_001     CA_1 2016-04-03      1    0.000000   
90880  HOBBIES_1_001     CA_1 2016-04-04      0    0.000000   
90930  HOBBIES_1_001     CA_1 2016-04-05      4    0.000000   
90980  HOBBIES_1_001     CA_1 2016-04-06      2    1.445935   
91030  HOBBIES_1_001     CA_1 2016-04-07      3    1.454710   
91080  HOBBIES_1_001     CA_1 2016-04-08      0    1.359349   
91130  HOBBIES_1_001     CA_1 2016-04-09      1    1.500111   
91180  HOBBIES_1_001     CA_1 2016-04-10      2    1.618795   
91230  HOBBIES_1_001     CA_1 2016-04-11      0    1.374040   
91280  HOBBIES_1_001     CA_1 2016-04-12      0    0.000000   
91330  HOBBIES_1_001     CA_1 2016-04-13      0    0.000000   
91380  HOBBIES_1_001     CA_1 2016-04-14      1    0.000000   
91430  HOBBIES_1_001     CA_1 2016-04-15      1    1.50

In [50]:
print([
    name for name in globals()
    if "pred" in name.lower()
])

['pred_v10', 'demand_prediction', 'quantity_prediction_log', 'quantity_prediction', 'final_prediction', 'demand_prediction_march', 'quantity_prediction_log_march', 'quantity_prediction_march', 'final_prediction_march']


In [56]:
# ------------------------------------------------------------
# Step 2: Create the 7-day forecast using daily predictions
# ------------------------------------------------------------

test_april = test_april.sort_values(
    ["item_id", "store_id", "date"]
).copy()

test_april["forecast_next_7_days"] = (
    test_april
    .groupby(["item_id", "store_id"])["prediction"]
    .transform(
        lambda x: x.shift(-1)
                  .rolling(7, min_periods=7)
                  .sum()
                  .shift(-6)
    )
)

print(
    test_april[
        [
            "item_id",
            "store_id",
            "date",
            "actual_next_7_days",
            "forecast_next_7_days"
        ]
    ].head(15)
)

             item_id store_id       date  actual_next_7_days  \
95525  HOBBIES_1_001     CA_1 2016-04-01                10.0   
95575  HOBBIES_1_001     CA_1 2016-04-02                11.0   
95625  HOBBIES_1_001     CA_1 2016-04-03                12.0   
90880  HOBBIES_1_001     CA_1 2016-04-04                12.0   
90930  HOBBIES_1_001     CA_1 2016-04-05                 8.0   
90980  HOBBIES_1_001     CA_1 2016-04-06                 6.0   
91030  HOBBIES_1_001     CA_1 2016-04-07                 4.0   
91080  HOBBIES_1_001     CA_1 2016-04-08                 5.0   
91130  HOBBIES_1_001     CA_1 2016-04-09                 7.0   
91180  HOBBIES_1_001     CA_1 2016-04-10                 5.0   
91230  HOBBIES_1_001     CA_1 2016-04-11                 6.0   
91280  HOBBIES_1_001     CA_1 2016-04-12                 7.0   
91330  HOBBIES_1_001     CA_1 2016-04-13                 8.0   
91380  HOBBIES_1_001     CA_1 2016-04-14                10.0   
91430  HOBBIES_1_001     CA_1 2016-04-15

In [57]:
# ------------------------------------------------------------
# Step 3: Calculate safety stock and reorder point
# ------------------------------------------------------------

# Historical demand variability for each product-store pair
inventory_stats = (
    train_april
    .groupby(["item_id", "store_id"])["sales"]
    .agg(
        demand_std="std"
    )
    .reset_index()
)

lead_time_days = 7
z_score = 1.645

# Safety stock for 7-day lead time
inventory_stats["safety_stock"] = (
    z_score
    * inventory_stats["demand_std"]
    * np.sqrt(lead_time_days)
)

# Attach safety stock to the April test data
test_april = test_april.merge(
    inventory_stats[
        ["item_id", "store_id", "safety_stock"]
    ],
    on=["item_id", "store_id"],
    how="left"
)

# Reorder point
test_april["reorder_point"] = (
    test_april["forecast_next_7_days"]
    + test_april["safety_stock"]
)

print(
    test_april[
        [
            "item_id",
            "store_id",
            "date",
            "forecast_next_7_days",
            "safety_stock",
            "reorder_point",
            "actual_next_7_days"
        ]
    ].head(15)
)

          item_id store_id       date  forecast_next_7_days  safety_stock  \
0   HOBBIES_1_001     CA_1 2016-04-01              5.645922      2.940366   
1   HOBBIES_1_001     CA_1 2016-04-02              5.760104      2.940366   
2   HOBBIES_1_001     CA_1 2016-04-03              7.378900      2.940366   
3   HOBBIES_1_001     CA_1 2016-04-04              8.752939      2.940366   
4   HOBBIES_1_001     CA_1 2016-04-05              8.752939      2.940366   
5   HOBBIES_1_001     CA_1 2016-04-06              7.307005      2.940366   
6   HOBBIES_1_001     CA_1 2016-04-07              5.852295      2.940366   
7   HOBBIES_1_001     CA_1 2016-04-08              5.994773      2.940366   
8   HOBBIES_1_001     CA_1 2016-04-09              6.118591      2.940366   
9   HOBBIES_1_001     CA_1 2016-04-10              6.269836      2.940366   
10  HOBBIES_1_001     CA_1 2016-04-11              4.895796      2.940366   
11  HOBBIES_1_001     CA_1 2016-04-12              6.257891      2.940366   

In [58]:
8# ------------------------------------------------------------
# Step 4: Evaluate the inventory policy
# ------------------------------------------------------------

# Remove rows where a complete 7-day future window is unavailable
inventory_eval = test_april.dropna(
    subset=[
        "actual_next_7_days",
        "reorder_point"
    ]
).copy()

# Stockout occurs when actual future demand exceeds
# the inventory available at the reorder point.
inventory_eval["stockout"] = (
    inventory_eval["actual_next_7_days"]
    > inventory_eval["reorder_point"]
).astype(int)

# How much demand was left uncovered?
inventory_eval["shortage"] = (
    inventory_eval["actual_next_7_days"]
    - inventory_eval["reorder_point"]
).clip(lower=0)

# How much inventory was left unused?
inventory_eval["excess_inventory"] = (
    inventory_eval["reorder_point"]
    - inventory_eval["actual_next_7_days"]
).clip(lower=0)

# Overall metrics
stockout_rate = inventory_eval["stockout"].mean()
service_level = 1 - stockout_rate
average_shortage = inventory_eval["shortage"].mean()
average_excess = inventory_eval["excess_inventory"].mean()

print(f"Service Level:       {service_level:.4f}")
print(f"Stockout Rate:       {stockout_rate:.4f}")
print(f"Average Shortage:    {average_shortage:.4f}")
print(f"Average Excess:      {average_excess:.4f}")

Service Level:       0.6941
Stockout Rate:       0.3059
Average Shortage:    0.5042
Average Excess:      3.0847


The initial inventory calculation appeared to achieve 100% service level under monthly aggregation, but after implementing a proper 7-day lead-time evaluation, the measured service level was 69.41%, revealing substantial stockout risk.

In [59]:
pair_performance = (
    inventory_eval
    .groupby(["item_id", "store_id"])
    .agg(
        service_level=("stockout", lambda x: 1 - x.mean()),
        stockout_rate=("stockout", "mean"),
        avg_shortage=("shortage", "mean"),
        avg_excess=("excess_inventory", "mean")
    )
    .reset_index()
    .sort_values("stockout_rate", ascending=False)
)

print(pair_performance)

          item_id store_id  service_level  stockout_rate  avg_shortage  \
27  HOBBIES_1_003     WI_1       0.117647       0.882353      1.588335   
28  HOBBIES_1_003     WI_2       0.117647       0.882353      1.030179   
23  HOBBIES_1_003     CA_4       0.176471       0.823529      0.585710   
21  HOBBIES_1_003     CA_2       0.176471       0.823529      2.123650   
4   HOBBIES_1_001     TX_1       0.176471       0.823529      0.300433   
26  HOBBIES_1_003     TX_3       0.235294       0.764706      0.791378   
48  HOBBIES_1_005     WI_2       0.235294       0.764706      2.815321   
18  HOBBIES_1_002     WI_2       0.294118       0.705882      1.474014   
8   HOBBIES_1_001     WI_2       0.352941       0.647059      0.691616   
38  HOBBIES_1_004     WI_2       0.352941       0.647059      1.832616   
24  HOBBIES_1_003     TX_1       0.352941       0.647059      1.243123   
6   HOBBIES_1_001     TX_3       0.352941       0.647059      0.692066   
20  HOBBIES_1_003     CA_1       0.352

In [60]:
inventory_eval_later = inventory_eval[
    inventory_eval["date"] >= "2016-04-08"
].copy()

stockout_rate_later = inventory_eval_later["stockout"].mean()
service_level_later = 1 - stockout_rate_later

average_shortage_later = inventory_eval_later["shortage"].mean()
average_excess_later = inventory_eval_later["excess_inventory"].mean()

print("Service Level:", service_level_later)
print("Stockout Rate:", stockout_rate_later)
print("Average Shortage:", average_shortage_later)
print("Average Excess:", average_excess_later)

Service Level: 0.6719999999999999
Stockout Rate: 0.328
Average Shortage: 0.6546251581221343
Average Excess: 3.13800609624284


We should test whether the forecast itself is biased for those bad pairs.

Specifically, let's inspect:

actual_next_7_days
vs
forecast_next_7_days

for the worst pairs such as:

HOBBIES_1_003 / WI_1
HOBBIES_1_003 / WI_2
HOBBIES_1_003 / CA_4
HOBBIES_1_003 / CA_2

If forecast ≈ actual but reorder point still fails → inventory formula problem.

If forecast << actual → forecast bias problem.

In [61]:
worst_pairs = [
    ("HOBBIES_1_003", "WI_1"),
    ("HOBBIES_1_003", "WI_2"),
    ("HOBBIES_1_003", "CA_4"),
    ("HOBBIES_1_003", "CA_2"),
    ("HOBBIES_1_001", "TX_1")
]

forecast_bias = (
    inventory_eval[
        inventory_eval[["item_id", "store_id"]]
        .apply(tuple, axis=1)
        .isin(worst_pairs)
    ]
    .groupby(["item_id", "store_id"])
    .apply(
        lambda x: pd.Series({
            "avg_actual": x["actual_next_7_days"].mean(),
            "avg_forecast": x["forecast_next_7_days"].mean(),
            "mean_error": (
                x["actual_next_7_days"] -
                x["forecast_next_7_days"]
            ).mean(),
            "mae": (
                x["actual_next_7_days"] -
                x["forecast_next_7_days"]
            ).abs().mean()
        }),
        include_groups=False
    )
    .reset_index()
)

print(forecast_bias)

         item_id store_id  avg_actual  avg_forecast  mean_error       mae
0  HOBBIES_1_001     TX_1    2.000000           0.0    2.000000  2.000000
1  HOBBIES_1_003     CA_2    3.294118           0.0    3.294118  3.294118
2  HOBBIES_1_003     CA_4    1.941176           0.0    1.941176  1.941176
3  HOBBIES_1_003     WI_1    3.000000           0.0    3.000000  3.000000
4  HOBBIES_1_003     WI_2    2.117647           0.0    2.117647  2.117647


In [63]:
[c for c in test_april.columns if "pred" in c.lower()]

['naive_prediction', 'prediction']

In [64]:
test_april[
    test_april["item_id"].isin([
        "HOBBIES_1_001",
        "HOBBIES_1_003"
    ])
    & test_april["store_id"].isin([
        "TX_1", "CA_2", "CA_4", "WI_1", "WI_2"
    ])
][
    ["item_id", "store_id", "date", "sales", "prediction"]
].head(50)

,item_id,store_id,date,sales,prediction
24,HOBBIES_1_001,CA_2,2016-04-01,1,0.000000
25,HOBBIES_1_001,CA_2,2016-04-02,1,1.483904
26,HOBBIES_1_001,CA_2,2016-04-03,0,1.450712
27,HOBBIES_1_001,CA_2,2016-04-04,2,0.000000
28,HOBBIES_1_001,CA_2,2016-04-05,0,0.000000
29,HOBBIES_1_001,CA_2,2016-04-06,0,0.000000
30,HOBBIES_1_001,CA_2,2016-04-07,0,0.000000
31,HOBBIES_1_001,CA_2,2016-04-08,0,0.000000
32,HOBBIES_1_001,CA_2,2016-04-09,0,0.000000
33,HOBBIES_1_001,CA_2,2016-04-10,1,0.000000


In [65]:
test_april[
    (test_april["item_id"] == "HOBBIES_1_003") &
    (test_april["store_id"].isin([
        "WI_1", "WI_2", "CA_4", "CA_2"
    ]))
][
    ["item_id", "store_id", "date", "sales", "prediction"]
].head(100)

,item_id,store_id,date,sales,prediction
504,HOBBIES_1_003,CA_2,2016-04-01,3,0.0
505,HOBBIES_1_003,CA_2,2016-04-02,1,0.0
506,HOBBIES_1_003,CA_2,2016-04-03,0,0.0
507,HOBBIES_1_003,CA_2,2016-04-04,0,0.0
508,HOBBIES_1_003,CA_2,2016-04-05,3,0.0
...,...,...,...,...,...
691,HOBBIES_1_003,WI_2,2016-04-20,0,0.0
692,HOBBIES_1_003,WI_2,2016-04-21,1,0.0
693,HOBBIES_1_003,WI_2,2016-04-22,0,0.0
694,HOBBIES_1_003,WI_2,2016-04-23,0,0.0


In [66]:
print(type(demand_prediction))
print(len(demand_prediction))

<class 'numpy.ndarray'>
1200


In [67]:
test_april["classifier_prediction"] = demand_prediction

test_april[
    (test_april["item_id"] == "HOBBIES_1_003") &
    (test_april["store_id"].isin(["CA_2", "CA_4", "WI_1", "WI_2"]))
][
    ["item_id", "store_id", "date", "sales", "classifier_prediction", "prediction"]
].head(100)

,item_id,store_id,date,sales,classifier_prediction,prediction
504,HOBBIES_1_003,CA_2,2016-04-01,3,0,0.0
505,HOBBIES_1_003,CA_2,2016-04-02,1,0,0.0
506,HOBBIES_1_003,CA_2,2016-04-03,0,0,0.0
507,HOBBIES_1_003,CA_2,2016-04-04,0,0,0.0
508,HOBBIES_1_003,CA_2,2016-04-05,3,0,0.0
...,...,...,...,...,...,...
691,HOBBIES_1_003,WI_2,2016-04-20,0,0,0.0
692,HOBBIES_1_003,WI_2,2016-04-21,1,0,0.0
693,HOBBIES_1_003,WI_2,2016-04-22,0,0,0.0
694,HOBBIES_1_003,WI_2,2016-04-23,0,0,0.0


In [69]:
(train_april[train_april["item_id"] == "HOBBIES_1_003"]["sales"] > 0).mean()

np.float64(0.05964535196131112)

| Metric    |        V12 |
| --------- | ---------: |
| Accuracy  |     71.92% |
| Precision |     72.59% |
| Recall    | **42.70%** |
| F1        |     53.77% |


In [70]:
print([x for x in globals() if "cls" in x.lower() or "classifier" in x.lower()])

['XGBClassifier', 'X_train_cls', 'y_train_cls', 'X_test_cls', 'y_test_cls', 'demand_classifier', 'classifier_accuracy', 'classifier_precision', 'classifier_recall', 'classifier_f1', 'X_train_cls_march', 'y_train_cls_march', 'X_test_cls_march', 'y_test_cls_march', 'demand_classifier_march']


In [71]:
test_april["demand_probability"] = demand_classifier.predict_proba(X_test_cls)[:, 1]

test_april[
    (test_april["item_id"] == "HOBBIES_1_003") &
    (test_april["store_id"].isin(["CA_2", "CA_4", "WI_1", "WI_2"]))
][
    ["item_id", "store_id", "date", "sales", "demand_probability", "classifier_prediction"]
].head(30)

,item_id,store_id,date,sales,demand_probability,classifier_prediction
504,HOBBIES_1_003,CA_2,2016-04-01,3,0.212716,0
505,HOBBIES_1_003,CA_2,2016-04-02,1,0.438729,0
506,HOBBIES_1_003,CA_2,2016-04-03,0,0.370218,0
507,HOBBIES_1_003,CA_2,2016-04-04,0,0.237268,0
508,HOBBIES_1_003,CA_2,2016-04-05,3,0.234846,0
509,HOBBIES_1_003,CA_2,2016-04-06,0,0.393192,0
510,HOBBIES_1_003,CA_2,2016-04-07,0,0.358567,0
511,HOBBIES_1_003,CA_2,2016-04-08,0,0.486769,0
512,HOBBIES_1_003,CA_2,2016-04-09,0,0.361956,0
513,HOBBIES_1_003,CA_2,2016-04-10,0,0.312965,0


In [72]:
from sklearn.metrics import precision_score, recall_score, f1_score

probabilities = demand_classifier.predict_proba(X_test_cls)[:, 1]

for threshold in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    pred = (probabilities >= threshold).astype(int)

    print(
        f"Threshold {threshold:.2f} | "
        f"Precision: {precision_score(y_test_cls, pred):.3f} | "
        f"Recall: {recall_score(y_test_cls, pred):.3f} | "
        f"F1: {f1_score(y_test_cls, pred):.3f}"
    )

Threshold 0.10 | Precision: 0.407 | Recall: 0.974 | F1: 0.574
Threshold 0.15 | Precision: 0.438 | Recall: 0.924 | F1: 0.594
Threshold 0.20 | Precision: 0.471 | Recall: 0.865 | F1: 0.610
Threshold 0.25 | Precision: 0.486 | Recall: 0.808 | F1: 0.607
Threshold 0.30 | Precision: 0.540 | Recall: 0.739 | F1: 0.624
Threshold 0.35 | Precision: 0.583 | Recall: 0.664 | F1: 0.621
Threshold 0.40 | Precision: 0.633 | Recall: 0.566 | F1: 0.598
Threshold 0.45 | Precision: 0.673 | Recall: 0.497 | F1: 0.571
Threshold 0.50 | Precision: 0.726 | Recall: 0.427 | F1: 0.538


In [74]:
threshold = 0.30

demand_prediction_30 = (
    demand_classifier.predict_proba(X_test_cls)[:, 1] >= threshold
).astype(int)

print("Predicted demand days:", demand_prediction_30.sum())
print("Total test days:", len(demand_prediction_30))

Predicted demand days: 628
Total test days: 1200


In [75]:
quantity_prediction_v12 = np.expm1(quantity_prediction_log)
quantity_prediction_v12 = np.clip(quantity_prediction_v12, 0, None)

final_prediction_30 = (
    quantity_prediction_v12 * demand_prediction_30
)

test_april["prediction_30"] = final_prediction_30

print(test_april["prediction_30"].describe())

count    1200.000000
mean        0.954145
std         1.144396
min         0.000000
25%         0.000000
50%         1.228270
75%         1.510334
max         7.639434
Name: prediction_30, dtype: float64


In [76]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae_30 = mean_absolute_error(
    test_april["sales"],
    test_april["prediction_30"]
)

rmse_30 = np.sqrt(
    mean_squared_error(
        test_april["sales"],
        test_april["prediction_30"]
    )
)

print("V12 threshold 0.30")
print("MAE :", mae_30)
print("RMSE:", rmse_30)

V12 threshold 0.30
MAE : 0.7706296567122142
RMSE: 1.1863225688178942


| V12                |        MAE |       RMSE |
| ------------------ | ---------: | ---------: |
| Threshold **0.50** | **0.6254** | **1.1461** |
| Threshold **0.30** |     0.7706 |     1.1863 |
| Change             | **+23.2%** |      +3.5% |


But there is an important twist: your objective is not just point-forecast accuracy. It's inventory performance. A model can have worse MAE while producing better reorder decisions, particularly when underprediction is more expensive than overprediction.

In [77]:
test_april["forecast_next_7_days_30"] = (
    test_april
    .groupby(["item_id", "store_id"])["prediction_30"]
    .transform(
        lambda x: x.shift(-1)
                  .rolling(7, min_periods=7)
                  .sum()
                  .shift(-6)
    )
)

In [78]:
test_april[
    [
        "date",
        "actual_next_7_days",
        "forecast_next_7_days",
        "forecast_next_7_days_30"
    ]
].head(15)

,date,actual_next_7_days,forecast_next_7_days,forecast_next_7_days_30
0,2016-04-01,10.0,5.645922,9.673717
1,2016-04-02,11.0,5.760104,9.787899
2,2016-04-03,12.0,7.378900,10.052713
3,2016-04-04,12.0,8.752939,10.090599
4,2016-04-05,8.0,8.752939,10.232997
5,2016-04-06,6.0,7.307005,10.330868
6,2016-04-07,4.0,5.852295,10.429801
7,2016-04-08,5.0,5.994773,10.572280
8,2016-04-09,7.0,6.118591,10.696097
9,2016-04-10,5.0,6.269836,10.847342


In [79]:
test_april["reorder_point_30"] = (
    test_april["forecast_next_7_days_30"]
    + test_april["safety_stock"]
)

In [80]:
inventory_eval_30 = test_april.dropna(
    subset=["actual_next_7_days", "reorder_point_30"]
).copy()

inventory_eval_30["stockout"] = (
    inventory_eval_30["actual_next_7_days"]
    > inventory_eval_30["reorder_point_30"]
).astype(int)

inventory_eval_30["shortage"] = (
    inventory_eval_30["actual_next_7_days"]
    - inventory_eval_30["reorder_point_30"]
).clip(lower=0)

inventory_eval_30["excess_inventory"] = (
    inventory_eval_30["reorder_point_30"]
    - inventory_eval_30["actual_next_7_days"]
).clip(lower=0)

print("Service Level:", 1 - inventory_eval_30["stockout"].mean())
print("Stockout Rate:", inventory_eval_30["stockout"].mean())
print("Average Shortage:", inventory_eval_30["shortage"].mean())
print("Average Excess:", inventory_eval_30["excess_inventory"].mean())

Service Level: 0.8741176470588236
Stockout Rate: 0.12588235294117647
Average Shortage: 0.1588374727315469
Average Excess: 5.738849525924167


That substantially reduces stockouts:

30.6% → 12.6%

and cuts the average shortage:

0.504 → 0.159 units

LETS TRY FOR 40

In [82]:
threshold = 0.40

demand_prediction_40 = (
    demand_classifier.predict_proba(X_test_cls)[:, 1] >= threshold
).astype(int)

quantity_prediction_v12 = np.expm1(quantity_prediction_log)
quantity_prediction_v12 = np.clip(quantity_prediction_v12, 0, None)

test_april["prediction_40"] = (
    quantity_prediction_v12 * demand_prediction_40
)

print("Predicted demand days:", demand_prediction_40.sum())

Predicted demand days: 411


| Threshold | Forecast MAE | Service level | Avg excess |
| --------- | -----------: | ------------: | ---------: |
| 0.50      |    **0.625** |        69.41% |       3.08 |
| 0.30      |        0.771 |    **87.41%** |       5.74 |


In [84]:
[x for x in globals() if "qty" in x.lower() or "quantity" in x.lower()]

['train_quantity',
 'X_train_quantity',
 'y_train_quantity',
 'quantity_model',
 'quantity_prediction_log',
 'quantity_prediction',
 'train_quantity_march',
 'X_train_quantity_march',
 'y_train_quantity_march',
 'quantity_model_march',
 'quantity_prediction_log_march',
 'quantity_prediction_march',
 'quantity_prediction_v12']

In [85]:
# ============================================
# FINAL V1 MODEL
# ============================================

demand_classifier_v1 = demand_classifier
quantity_model_v1 = quantity_model

print("Final V1 models created successfully.")
print("Classifier:", type(demand_classifier_v1).__name__)
print("Quantity model:", type(quantity_model_v1).__name__)

Final V1 models created successfully.
Classifier: XGBClassifier
Quantity model: XGBRegressor


In [86]:
FEATURES_V1 = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "rolling_max_7",
    "rolling_max_28",
    "rolling_std_7",
    "trend",
    "dayofweek",
    "month",
    "weekofyear",
    "is_weekend",
    "has_event",
    "state_encoded",
    "store_encoded",
    "item_encoded"
]

print("Final V1 feature count:", len(FEATURES_V1))

Final V1 feature count: 18


In [87]:
import joblib
from pathlib import Path

# Create models directory if it doesn't exist
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

# Save final V1 models
joblib.dump(
    demand_classifier_v1,
    models_dir / "demand_classifier_v1.pkl"
)

joblib.dump(
    quantity_model_v1,
    models_dir / "quantity_model_v1.pkl"
)

print("V1 models saved successfully.")

V1 models saved successfully.


In [88]:
joblib.dump(
    FEATURES_V1,
    models_dir / "features_v1.pkl"
)

print("V1 feature list saved successfully.")

V1 feature list saved successfully.


In [89]:
import joblib
from pathlib import Path

# Recreate V1 inventory statistics
inventory_stats_v1 = (
    train_april
    .groupby(["item_id", "store_id"])["sales"]
    .agg(demand_std="std")
    .reset_index()
)

# V1 inventory settings
LEAD_TIME_DAYS_V1 = 7
Z_SCORE_V1 = 1.645

inventory_stats_v1["safety_stock"] = (
    Z_SCORE_V1
    * inventory_stats_v1["demand_std"]
    * np.sqrt(LEAD_TIME_DAYS_V1)
)

# Save
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(
    inventory_stats_v1,
    models_dir / "inventory_stats_v1.pkl"
)

print("Inventory statistics saved.")
print(f"Pairs: {len(inventory_stats_v1)}")

Inventory statistics saved.
Pairs: 50
